# Notebook 02 — Corruption Module
## Adaptive Reliability-Aware Fusion (ARAF) Project

**Goal of this notebook:**  
Build the corruption module — the component that deliberately degrades inputs
during training so the model learns to be robust when real-world inputs are
noisy, blurry, incomplete, or missing entirely.

**Why this is the most important notebook:**  
Standard multimodal models assume clean inputs at test time. In the real world,
images can be blurry, text can be garbled, and sometimes an entire modality is
missing. ARAF is designed to handle this — but only if we train it on corrupted
data. The corruption module is what makes that possible.

**What you will learn:**
- What types of corruption exist and why each one matters
- How to implement image corruptions (noise, blur, occlusion)
- How to implement text corruptions (dropout, shuffling, masking)
- How to simulate missing modalities
- How severity levels work
- How the corruption module slots into the `MultimodalSample` contract

**Key design principle:**  
The corruption module takes a clean `MultimodalSample` and returns a corrupted
one. The label never changes. Only the input tensors change.

---


## 1. Imports and setup

We load everything from Notebook 01 that we need here. In a real project this
would be an `import` from a `.py` file — for now we re-run the essentials.


In [ ]:
import os
import json
import copy
import random
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional, List, Dict, Tuple, Callable

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image, ImageFilter

import torch
import torch.nn.functional as F
from torchvision import transforms
from transformers import BertTokenizer

# ── Reproducibility ───────────────────────────────────────────────────────────
# Setting seeds ensures corruptions are reproducible for debugging.
# During training we will NOT fix the seed so corruptions are random each epoch.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Imports successful.")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")


## 2. Reload `MultimodalSample` and grab a few clean samples

We redefine `MultimodalSample` here (same as Notebook 01) and load a few clean
samples from VQA v2 to use as inputs for testing our corruption functions.

**Why reload instead of import?**  
Because we're still in notebook-land. In Notebook 05 (training) we will import
from `.py` files. For now, redefining keeps each notebook self-contained.


In [ ]:
# ── Re-define MultimodalSample (identical to Notebook 01) ────────────────────
@dataclass
class MultimodalSample:
    image: torch.Tensor
    text_ids: torch.Tensor
    attention_mask: torch.Tensor
    label: torch.Tensor
    raw_image: Optional[object] = None
    raw_text: str = ""
    dataset_name: str = "vqa_v2"
    sample_id: str = ""
    image_corrupted: bool = False
    text_corrupted: bool = False
    image_missing: bool = False
    text_missing: bool = False
    corruption_severity: float = 0.0

# ── Image constants ───────────────────────────────────────────────────────────
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
IMAGE_SIZE    = 224
MAX_TEXT_LEN  = 32

clean_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

def denormalize(tensor: torch.Tensor) -> np.ndarray:
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std  = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    img  = (tensor * std + mean).clamp(0, 1)
    return (img * 255).byte().permute(1, 2, 0).numpy()

print("MultimodalSample and transforms defined.")


In [ ]:
from datasets import load_dataset

# Load a small slice of VQA v2 — just enough to test corruptions visually
print("Loading a few VQA v2 samples for corruption testing...")
hf_val_full = load_dataset("lmms-lab/VQAv2", split="validation",
                           trust_remote_code=True)

# Load answer vocab saved in Notebook 01
with open("answer_vocab.json") as f:
    answer2idx = json.load(f)

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

# ── Minimal dataset loader (same logic as Notebook 01) ───────────────────────
def load_sample(row) -> MultimodalSample:
    """Load one HuggingFace VQA row into a MultimodalSample."""
    from collections import Counter
    pil = row["image"].convert("RGB")
    img = clean_transform(pil)
    enc = tokenizer(row["question"], padding="max_length",
                    max_length=MAX_TEXT_LEN, truncation=True,
                    return_tensors="pt")
    # soft label
    num_classes = len(answer2idx)
    label = torch.zeros(num_classes)
    cnt = Counter(a["answer"].lower().strip() for a in row["answers"])
    for ans, c in cnt.items():
        if ans in answer2idx:
            label[answer2idx[ans]] = min(c / 3.0, 1.0)

    return MultimodalSample(
        image=img, text_ids=enc["input_ids"].squeeze(0),
        attention_mask=enc["attention_mask"].squeeze(0),
        label=label, raw_image=pil, raw_text=row["question"],
        dataset_name="vqa_v2",
        sample_id=str(row.get("question_id", 0)),
    )

# Load 8 clean samples for testing
clean_samples = [load_sample(hf_val_full[i]) for i in range(8)]
print(f"Loaded {len(clean_samples)} clean samples.")
print(f"Sample 0 question : '{clean_samples[0].raw_text}'")
print(f"Image shape       : {clean_samples[0].image.shape}")
print(f"Text IDs shape    : {clean_samples[0].text_ids.shape}")


## 3. Severity levels — the key design decision

Every corruption in our module takes a `severity` parameter from 1 to 5.

| Level | Meaning | Effect |
|---|---|---|
| 1 | Barely perceptible | Slight noise, 1-2 tokens dropped |
| 2 | Mild | Noticeable but manageable |
| 3 | Moderate | Clearly degraded |
| 4 | Severe | Hard for humans too |
| 5 | Extreme | Almost completely destroyed |

**Why 5 levels?**  
It lets us plot robustness curves: how does model accuracy degrade as severity
increases? This is a standard evaluation protocol in robustness research
(e.g. ImageNet-C uses 5 severity levels for the same reason).

**Why not just a continuous 0-1 scale?**  
Discrete levels make results reproducible and comparable across papers.
"Severity 3" is a specific, well-defined thing. "Severity 0.47" is not.


In [ ]:
# Severity → concrete parameter mappings
# These are the actual values used by each corruption function.
# We define them centrally so they're easy to tune.

SEVERITY_PARAMS = {
    "gaussian_noise": {
        # std of noise added to normalized image tensor
        1: 0.05, 2: 0.10, 3: 0.20, 4: 0.35, 5: 0.50
    },
    "motion_blur": {
        # kernel size (must be odd)
        1: 3, 2: 5, 3: 7, 4: 11, 5: 15
    },
    "occlusion": {
        # fraction of image area covered by black patches
        1: 0.05, 2: 0.10, 3: 0.20, 4: 0.35, 5: 0.50
    },
    "token_dropout": {
        # fraction of non-special tokens replaced with [PAD]
        1: 0.10, 2: 0.20, 3: 0.35, 4: 0.50, 5: 0.70
    },
    "token_shuffle": {
        # fraction of non-special tokens randomly shuffled
        1: 0.15, 2: 0.30, 3: 0.50, 4: 0.70, 5: 1.00
    },
    "token_mask": {
        # fraction of non-special tokens replaced with [MASK]
        1: 0.10, 2: 0.20, 3: 0.35, 4: 0.50, 5: 0.70
    },
}

print("Severity parameter table:")
for corruption, levels in SEVERITY_PARAMS.items():
    print(f"  {corruption:20s}: {levels}")


## 4. Image corruptions

We implement three image corruption types. Each takes a normalized image tensor
`[3, 224, 224]` and returns a corrupted tensor of the same shape.

### Why these three?

- **Gaussian noise** — simulates sensor noise, low-light photography, compression
  artifacts. Very common in real deployments.
- **Motion blur** — simulates camera shake, fast-moving subjects. Common in
  mobile photos and video frames.
- **Occlusion** — simulates objects blocking part of the image, watermarks,
  cropping errors, or partial visibility. This is the hardest for models because
  information is completely removed, not just degraded.

### Important: corruptions operate on normalized tensors

Our images are already normalized (values roughly in [-2, 2]). We add noise
directly in this normalized space. This is correct — it mimics how sensor noise
affects the signal before any processing.


In [ ]:
def corrupt_gaussian_noise(image: torch.Tensor, severity: int) -> torch.Tensor:
    """
    Add Gaussian noise to a normalized image tensor.
    
    Why Gaussian? Real sensor noise is approximately Gaussian distributed
    (central limit theorem — many small independent noise sources sum to
    a Gaussian). It's the most realistic and most studied noise type.
    
    Args:
        image    : [3, H, W] normalized float32 tensor
        severity : 1-5
    Returns:
        Corrupted tensor, same shape, clamped to reasonable range
    """
    std   = SEVERITY_PARAMS["gaussian_noise"][severity]
    noise = torch.randn_like(image) * std
    # Clamp to ~4 std of ImageNet normalized range to avoid extreme values
    return (image + noise).clamp(-3.0, 3.0)


def corrupt_motion_blur(image: torch.Tensor, severity: int) -> torch.Tensor:
    """
    Apply motion blur using a horizontal averaging kernel.
    
    Why a horizontal kernel? Motion blur is typically directional.
    Horizontal is the most common direction (left-right camera movement).
    For a research project, one direction is sufficient — you could extend
    this to random angle motion blur as a future improvement.
    
    Implementation: we use F.conv2d with a 1D averaging kernel. This is
    faster and more differentiable than PIL's ImageFilter.
    
    Args:
        image    : [3, H, W] normalized float32 tensor
        severity : 1-5
    Returns:
        Blurred tensor, same shape
    """
    k = SEVERITY_PARAMS["motion_blur"][severity]  # kernel size (odd number)
    
    # Build horizontal motion blur kernel: [1, 1, 1, k] averaged
    kernel = torch.ones(1, 1, 1, k) / k  # shape [1, 1, 1, k]
    kernel = kernel.expand(3, 1, 1, k)   # apply same kernel to all 3 channels
    
    # Pad to maintain spatial dimensions
    pad    = k // 2
    img    = image.unsqueeze(0)                      # [1, 3, H, W] for conv2d
    blurred = F.conv2d(img, kernel, padding=(0, pad), groups=3)
    return blurred.squeeze(0)                        # back to [3, H, W]


def corrupt_occlusion(image: torch.Tensor, severity: int,
                      num_patches: int = 3) -> torch.Tensor:
    """
    Apply random rectangular occlusion patches (set to zero = black).
    
    Why zero? In normalized space, zero corresponds roughly to the ImageNet
    mean color (gray). We could also use random noise patches, but zero
    (gray fill) is the standard in robustness literature (CutOut, Random
    Erasing) because it's a neutral, non-informative fill.
    
    Why multiple patches instead of one big one?
    Multiple small patches are harder for the model to reason around than
    one large patch. It more realistically simulates real occlusion
    (e.g. multiple objects blocking parts of a scene).
    
    Args:
        image      : [3, H, W] normalized float32 tensor
        severity   : 1-5
        num_patches: number of occlusion rectangles to apply
    Returns:
        Occluded tensor, same shape
    """
    fraction = SEVERITY_PARAMS["occlusion"][severity]
    img      = image.clone()
    _, H, W  = img.shape
    
    # Total area to occlude, split across num_patches patches
    total_area  = int(H * W * fraction)
    patch_area  = total_area // num_patches
    patch_side  = int(patch_area ** 0.5)  # square-ish patches
    patch_side  = max(patch_side, 4)       # minimum 4px
    
    for _ in range(num_patches):
        # Random top-left corner
        top  = random.randint(0, max(0, H - patch_side))
        left = random.randint(0, max(0, W - patch_side))
        # Zero out the patch (all channels)
        img[:, top:top+patch_side, left:left+patch_side] = 0.0
    
    return img


print("Image corruption functions defined.")
print("Testing on sample 0...")

s = clean_samples[0]
for sev in [1, 3, 5]:
    noisy   = corrupt_gaussian_noise(s.image, sev)
    blurred = corrupt_motion_blur(s.image, sev)
    occluded= corrupt_occlusion(s.image, sev)
    print(f"  Severity {sev}: noise std={noisy.std():.3f}, "
          f"blur diff={((s.image-blurred)**2).mean():.4f}, "
          f"occluded zeros={(occluded==0).float().mean():.3f}")


## 5. Visualize image corruptions

Always look at your corruptions before using them. This is the most important
debugging step — if a corruption looks wrong visually, it will produce wrong
training signal.

We display all three corruption types across all 5 severity levels for one
sample image.


In [ ]:
def visualize_image_corruptions(sample: MultimodalSample):
    """Show clean vs all corruption types across all severity levels."""
    corruption_fns = {
        "Gaussian noise" : corrupt_gaussian_noise,
        "Motion blur"    : corrupt_motion_blur,
        "Occlusion"      : corrupt_occlusion,
    }
    severities = [1, 2, 3, 4, 5]
    
    n_rows = len(corruption_fns) + 1  # +1 for clean row
    n_cols = len(severities) + 1      # +1 for label column
    
    fig, axes = plt.subplots(n_rows, n_cols,
                             figsize=(n_cols * 2.5, n_rows * 2.5))
    
    # ── Row 0: clean image repeated ───────────────────────────────────────────
    axes[0, 0].text(0.5, 0.5, "Clean", ha="center", va="center",
                    fontsize=11, fontweight="bold",
                    transform=axes[0, 0].transAxes)
    axes[0, 0].axis("off")
    for j in range(len(severities)):
        axes[0, j+1].imshow(denormalize(sample.image))
        axes[0, j+1].set_title(f"Severity {severities[j]}", fontsize=9)
        axes[0, j+1].axis("off")
    
    # ── Rows 1+: corrupted versions ───────────────────────────────────────────
    for i, (name, fn) in enumerate(corruption_fns.items()):
        row = i + 1
        axes[row, 0].text(0.5, 0.5, name, ha="center", va="center",
                          fontsize=9, fontweight="bold",
                          transform=axes[row, 0].transAxes)
        axes[row, 0].axis("off")
        for j, sev in enumerate(severities):
            corrupted = fn(sample.image, sev)
            axes[row, j+1].imshow(denormalize(corrupted))
            axes[row, j+1].axis("off")
    
    q = sample.raw_text[:50] + "..." if len(sample.raw_text) > 50 else sample.raw_text
    fig.suptitle(f"Image corruptions
Q: '{q}'", fontsize=11, y=1.01)
    plt.tight_layout()
    plt.savefig("image_corruptions.png", dpi=100, bbox_inches="tight")
    plt.show()
    print("Saved to image_corruptions.png")

visualize_image_corruptions(clean_samples[0])


## 6. Text corruptions

Text corruptions operate on `text_ids` — the tokenized integer tensor, not the
raw string. This is important: we corrupt *after* tokenization, which is what
the model actually sees.

### Why three types?

- **Token dropout** — randomly removes tokens (replaces with `[PAD]`). Simulates
  incomplete transcription, OCR errors, or network packet loss in streaming text.
- **Token shuffle** — randomly reorders tokens. Simulates word-order errors from
  bad translation systems, speech recognition errors, or noisy OCR.
- **Token mask** — replaces tokens with `[MASK]` (BERT's mask token). This is
  the most principled corruption: it forces the model to reason about what was
  there without seeing it. Also directly related to how BERT was pretrained.

### Special tokens are never corrupted

`[CLS]` (101), `[SEP]` (102), and `[PAD]` (0) tokens are never touched.
Corrupting them would break the sentence structure that BERT expects and cause
the encoder to produce garbage features regardless of what we do.


In [ ]:
# Special token IDs — these are NEVER corrupted
CLS_ID  = 101
SEP_ID  = 102
PAD_ID  = 0
MASK_ID = 103

def get_corruptible_positions(text_ids: torch.Tensor) -> torch.Tensor:
    """
    Return indices of tokens that CAN be corrupted.
    Excludes [CLS], [SEP], [PAD] tokens.
    
    Args:
        text_ids: [seq_len] int64 tensor
    Returns:
        1D tensor of valid indices
    """
    special = {CLS_ID, SEP_ID, PAD_ID}
    mask    = torch.ones(len(text_ids), dtype=torch.bool)
    for i, tid in enumerate(text_ids.tolist()):
        if tid in special:
            mask[i] = False
    return mask.nonzero(as_tuple=True)[0]


def corrupt_token_dropout(text_ids: torch.Tensor,
                          attention_mask: torch.Tensor,
                          severity: int) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Replace a fraction of non-special tokens with [PAD].
    Also sets attention_mask to 0 for dropped tokens
    (the encoder should ignore them).
    
    Args:
        text_ids      : [seq_len] int64
        attention_mask: [seq_len] int64
        severity      : 1-5
    Returns:
        (corrupted_ids, corrupted_mask) — same shapes as inputs
    """
    fraction    = SEVERITY_PARAMS["token_dropout"][severity]
    ids         = text_ids.clone()
    amask       = attention_mask.clone()
    valid_pos   = get_corruptible_positions(ids)
    
    if len(valid_pos) == 0:
        return ids, amask
    
    n_drop      = max(1, int(len(valid_pos) * fraction))
    drop_idx    = valid_pos[torch.randperm(len(valid_pos))[:n_drop]]
    
    ids[drop_idx]   = PAD_ID
    amask[drop_idx] = 0      # tell encoder to ignore these positions
    return ids, amask


def corrupt_token_shuffle(text_ids: torch.Tensor,
                          attention_mask: torch.Tensor,
                          severity: int) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Randomly shuffle a fraction of non-special tokens.
    Attention mask is unchanged (shuffled tokens are still real tokens).
    
    Args:
        text_ids      : [seq_len] int64
        attention_mask: [seq_len] int64
        severity      : 1-5
    Returns:
        (corrupted_ids, attention_mask) — attention mask unchanged
    """
    fraction  = SEVERITY_PARAMS["token_shuffle"][severity]
    ids       = text_ids.clone()
    valid_pos = get_corruptible_positions(ids)
    
    if len(valid_pos) == 0:
        return ids, attention_mask
    
    n_shuffle  = max(1, int(len(valid_pos) * fraction))
    shuf_idx   = valid_pos[torch.randperm(len(valid_pos))[:n_shuffle]]
    
    # Shuffle the selected positions amongst themselves
    shuffled_order      = shuf_idx[torch.randperm(len(shuf_idx))]
    ids[shuf_idx]       = ids[shuffled_order].clone()
    return ids, attention_mask


def corrupt_token_mask(text_ids: torch.Tensor,
                       attention_mask: torch.Tensor,
                       severity: int) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Replace a fraction of non-special tokens with [MASK].
    Attention mask is unchanged ([MASK] is a real token the encoder attends to).
    
    This is the most principled corruption: [MASK] tells the encoder
    'something was here but you cannot see it.' The encoder must infer
    meaning from context, which is exactly what BERT was trained to do.
    
    Args:
        text_ids      : [seq_len] int64
        attention_mask: [seq_len] int64
        severity      : 1-5
    Returns:
        (corrupted_ids, attention_mask)
    """
    fraction  = SEVERITY_PARAMS["token_mask"][severity]
    ids       = text_ids.clone()
    valid_pos = get_corruptible_positions(ids)
    
    if len(valid_pos) == 0:
        return ids, attention_mask
    
    n_mask    = max(1, int(len(valid_pos) * fraction))
    mask_idx  = valid_pos[torch.randperm(len(valid_pos))[:n_mask]]
    ids[mask_idx] = MASK_ID
    return ids, attention_mask


print("Text corruption functions defined.")


## 7. Visualize text corruptions

Unlike image corruptions, text corruptions are best visualized by decoding the
token IDs back to strings. We show the original question and each corrupted
version across severity levels.


In [ ]:
def visualize_text_corruptions(sample: MultimodalSample, tokenizer):
    """
    Display original text and all corrupted versions across severity levels.
    Decodes token IDs back to readable strings for inspection.
    """
    corruption_fns = {
        "Token dropout" : corrupt_token_dropout,
        "Token shuffle" : corrupt_token_shuffle,
        "Token mask"    : corrupt_token_mask,
    }
    severities = [1, 2, 3, 4, 5]
    
    print(f"Original question: '{sample.raw_text}'")
    print(f"Original token IDs (non-padding): "
          f"{sample.text_ids[sample.attention_mask.bool()].tolist()}")
    print()
    
    for name, fn in corruption_fns.items():
        print(f"{'─'*60}")
        print(f"Corruption: {name}")
        for sev in severities:
            c_ids, c_mask = fn(sample.text_ids, sample.attention_mask, sev)
            # Decode back to string (skip special tokens for readability)
            decoded = tokenizer.decode(c_ids, skip_special_tokens=False)
            # Count how many tokens were changed
            n_changed = (c_ids != sample.text_ids).sum().item()
            print(f"  Severity {sev} ({n_changed:2d} tokens changed): {decoded}")
        print()

visualize_text_corruptions(clean_samples[0], tokenizer)


## 8. Missing modality simulation

This is conceptually the simplest but most impactful corruption: we completely
remove one or both modalities.

### Why simulate missing modalities?

In practice:
- A user might send a meme with no caption
- An image might fail to load (network error)
- OCR might completely fail on a noisy image
- A sensor might disconnect during data collection

A robust model must handle these cases gracefully. If we never train on missing
modalities, the model will produce nonsense when one is absent at test time.

### How we simulate it

We replace the missing modality's tensor with **zeros**. This is the standard
approach because:
- Zero is a neutral, uninformative value
- The reliability estimator will learn to detect zero-filled tensors
- We set the `image_missing` / `text_missing` flags so the model knows during
  training — at test time, it must detect this from the tensor alone

### What the model learns

During training, whenever the image is zeroed out, the correct behavior is to
rely entirely on the text. The reliability estimator learns: "zero image =
unreliable image = weight text much more heavily." This generalizes to test
time even without the explicit flag.


In [ ]:
def corrupt_missing_modality(
    sample: MultimodalSample,
    drop_image: bool = False,
    drop_text: bool = False,
) -> MultimodalSample:
    """
    Simulate missing modalities by zeroing out the relevant tensor.
    
    Args:
        sample     : clean MultimodalSample
        drop_image : if True, replace image tensor with zeros
        drop_text  : if True, replace text_ids with [PAD] and mask with 0
    Returns:
        New MultimodalSample with missing modality zeroed out and flags set.
        Label is NEVER changed.
    """
    # Deep copy so we never mutate the original
    result = copy.deepcopy(sample)
    
    if drop_image:
        result.image         = torch.zeros_like(sample.image)
        result.image_missing = True
        result.image_corrupted = True
    
    if drop_text:
        # Zero out all non-CLS tokens: replace with PAD, mask=0
        result.text_ids        = torch.full_like(sample.text_ids, PAD_ID)
        result.attention_mask  = torch.zeros_like(sample.attention_mask)
        # Keep CLS token so encoder doesn't completely break
        result.text_ids[0]        = CLS_ID
        result.attention_mask[0]  = 1
        result.text_missing       = True
        result.text_corrupted     = True
    
    # Severity is 1.0 (maximum) for missing modality
    result.corruption_severity = 1.0
    
    return result


# ── Demonstrate ───────────────────────────────────────────────────────────────
s = clean_samples[0]

missing_image = corrupt_missing_modality(s, drop_image=True)
missing_text  = corrupt_missing_modality(s, drop_text=True)
missing_both  = corrupt_missing_modality(s, drop_image=True, drop_text=True)

print(f"Original       : image_missing={s.image_missing}, "
      f"text_missing={s.text_missing}")
print(f"Missing image  : image_missing={missing_image.image_missing}, "
      f"image sum={missing_image.image.sum():.1f} (should be 0)")
print(f"Missing text   : text_missing={missing_text.text_missing}, "
      f"non-pad tokens={missing_text.attention_mask.sum().item()} (should be 1 = CLS only)")
print(f"Missing both   : image_missing={missing_both.image_missing}, "
      f"text_missing={missing_both.text_missing}")
print(f"Label unchanged: {(s.label == missing_both.label).all().item()}")


## 9. Visualize missing modality


In [ ]:
def visualize_missing_modality(sample: MultimodalSample, tokenizer):
    """Show what missing modality looks like visually."""
    cases = {
        "Clean"         : sample,
        "Image missing" : corrupt_missing_modality(sample, drop_image=True),
        "Text missing"  : corrupt_missing_modality(sample, drop_text=True),
        "Both missing"  : corrupt_missing_modality(sample, drop_image=True,
                                                   drop_text=True),
    }
    
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    for ax, (name, s) in zip(axes, cases.items()):
        ax.imshow(denormalize(s.image))
        decoded = tokenizer.decode(s.text_ids, skip_special_tokens=True)
        decoded = decoded[:40] + "..." if len(decoded) > 40 else decoded
        ax.set_title(f"{name}\n'{decoded}'", fontsize=9)
        ax.axis("off")
    
    plt.suptitle("Missing modality simulation", fontsize=12)
    plt.tight_layout()
    plt.savefig("missing_modality.png", dpi=100, bbox_inches="tight")
    plt.show()
    print("Saved to missing_modality.png")

visualize_missing_modality(clean_samples[0], tokenizer)


## 10. The `CorruptionModule` class — putting it all together

Now we wrap everything into a single class. This is the interface the training
loop will use. It takes a clean `MultimodalSample` and returns a corrupted one,
randomly selecting the corruption type and severity according to configurable
probabilities.

### Key design decisions

**`p_corrupt_image` and `p_corrupt_text`**: Independent probabilities. Setting
both to 0.5 means each modality has a 50% chance of being corrupted each time.
This creates four possible training scenarios:
- Both clean (25% of the time)
- Image corrupted only (25%)
- Text corrupted only (25%)
- Both corrupted (25%)

This balanced mix is important: if we only ever corrupt both simultaneously,
the model never learns which individual modality is unreliable.

**`p_missing`**: Separate probability for the more extreme missing modality
case. Kept lower than `p_corrupt` because missing modality is a special case.

**`severity` can be fixed or random**: During training we typically sample
severity uniformly from 1-5. For evaluation we fix it to test specific levels.


In [ ]:
class CorruptionModule:
    """
    Applies configurable corruptions to MultimodalSample objects.
    
    Used in two modes:
    1. Training mode: random corruption type, random severity, random modality
    2. Evaluation mode: fixed corruption type and severity for systematic testing
    
    Args:
        p_corrupt_image : probability of corrupting the image modality
        p_corrupt_text  : probability of corrupting the text modality
        p_missing_image : probability of dropping image entirely
        p_missing_text  : probability of dropping text entirely
        severity        : fixed severity (1-5), or None for random
        image_corruptions: list of image corruption types to sample from
        text_corruptions : list of text corruption types to sample from
    """
    
    # Available corruption functions
    IMAGE_CORRUPTIONS = {
        "gaussian_noise" : corrupt_gaussian_noise,
        "motion_blur"    : corrupt_motion_blur,
        "occlusion"      : corrupt_occlusion,
    }
    TEXT_CORRUPTIONS = {
        "token_dropout"  : corrupt_token_dropout,
        "token_shuffle"  : corrupt_token_shuffle,
        "token_mask"     : corrupt_token_mask,
    }
    
    def __init__(
        self,
        p_corrupt_image   : float = 0.5,
        p_corrupt_text    : float = 0.5,
        p_missing_image   : float = 0.1,
        p_missing_text    : float = 0.1,
        severity          : Optional[int] = None,   # None = random 1-5
        image_corruptions : Optional[List[str]] = None,
        text_corruptions  : Optional[List[str]] = None,
    ):
        self.p_corrupt_image   = p_corrupt_image
        self.p_corrupt_text    = p_corrupt_text
        self.p_missing_image   = p_missing_image
        self.p_missing_text    = p_missing_text
        self.fixed_severity    = severity
        
        self.image_corruptions = image_corruptions or list(self.IMAGE_CORRUPTIONS)
        self.text_corruptions  = text_corruptions  or list(self.TEXT_CORRUPTIONS)
    
    def _sample_severity(self) -> int:
        """Return a severity level (fixed or random 1-5)."""
        if self.fixed_severity is not None:
            return self.fixed_severity
        return random.randint(1, 5)
    
    def __call__(self, sample: MultimodalSample) -> MultimodalSample:
        """
        Apply corruption to a clean sample. Returns a new MultimodalSample.
        The original sample is never mutated.
        """
        result = copy.deepcopy(sample)
        sev    = self._sample_severity()
        
        # ── Step 1: Missing modality (checked first, higher priority) ─────────
        drop_img  = random.random() < self.p_missing_image
        drop_text = random.random() < self.p_missing_text
        
        if drop_img or drop_text:
            result = corrupt_missing_modality(result,
                                              drop_image=drop_img,
                                              drop_text=drop_text)
            # If modality is missing, don't also corrupt it
            if drop_img and drop_text:
                return result
        
        # ── Step 2: Image corruption ──────────────────────────────────────────
        if not result.image_missing and random.random() < self.p_corrupt_image:
            corruption_name = random.choice(self.image_corruptions)
            corruption_fn   = self.IMAGE_CORRUPTIONS[corruption_name]
            result.image    = corruption_fn(result.image, sev)
            result.image_corrupted    = True
            result.corruption_severity = sev / 5.0  # normalize to [0,1]
        
        # ── Step 3: Text corruption ───────────────────────────────────────────
        if not result.text_missing and random.random() < self.p_corrupt_text:
            corruption_name = random.choice(self.text_corruptions)
            corruption_fn   = self.TEXT_CORRUPTIONS[corruption_name]
            result.text_ids, result.attention_mask = corruption_fn(
                result.text_ids, result.attention_mask, sev
            )
            result.text_corrupted      = True
            result.corruption_severity = max(result.corruption_severity, sev / 5.0)
        
        return result
    
    def corrupt_batch(self, samples: List[MultimodalSample]
                      ) -> List[MultimodalSample]:
        """Apply corruption to a list of samples (used in training loop)."""
        return [self(s) for s in samples]
    
    def get_eval_config(self, corruption_type: str, severity: int) -> 'CorruptionModule':
        """
        Return a deterministic corruption module for evaluation.
        Always applies the specified corruption at the specified severity.
        Used in Notebook 06 for systematic robustness evaluation.
        """
        is_image = corruption_type in self.IMAGE_CORRUPTIONS
        return CorruptionModule(
            p_corrupt_image   = 1.0 if is_image else 0.0,
            p_corrupt_text    = 0.0 if is_image else 1.0,
            p_missing_image   = 0.0,
            p_missing_text    = 0.0,
            severity          = severity,
            image_corruptions = [corruption_type] if is_image else None,
            text_corruptions  = [corruption_type] if not is_image else None,
        )


print("CorruptionModule class defined.")


## 11. Test the `CorruptionModule`

We run the module on a batch of clean samples and verify the output statistics.


In [ ]:
# ── Test in training mode (random corruptions) ────────────────────────────────
print("Testing CorruptionModule in training mode...")
print("(p_corrupt_image=0.5, p_corrupt_text=0.5, p_missing=0.1)")
print()

corruption_module = CorruptionModule(
    p_corrupt_image = 0.5,
    p_corrupt_text  = 0.5,
    p_missing_image = 0.1,
    p_missing_text  = 0.1,
    severity        = None,  # random
)

# Run 100 samples through and collect statistics
n_test          = 100
n_img_corrupted = 0
n_txt_corrupted = 0
n_img_missing   = 0
n_txt_missing   = 0
n_both_clean    = 0

for i in range(n_test):
    # Use modulo to cycle through our 8 clean samples
    s   = clean_samples[i % len(clean_samples)]
    out = corruption_module(s)
    
    n_img_corrupted += out.image_corrupted
    n_txt_corrupted += out.text_corrupted
    n_img_missing   += out.image_missing
    n_txt_missing   += out.text_missing
    n_both_clean    += (not out.image_corrupted and not out.text_corrupted)
    
    # Verify label never changes
    assert (out.label == s.label).all(), "Label was changed — bug!"
    # Verify shapes are preserved
    assert out.image.shape == s.image.shape, "Image shape changed — bug!"
    assert out.text_ids.shape == s.text_ids.shape, "Text shape changed — bug!"

print(f"Results over {n_test} samples:")
print(f"  Image corrupted or missing : {n_img_corrupted} ({n_img_corrupted}%)")
print(f"  Text corrupted or missing  : {n_txt_corrupted} ({n_txt_corrupted}%)")
print(f"  Image missing              : {n_img_missing} ({n_img_missing}%)")
print(f"  Text missing               : {n_txt_missing} ({n_txt_missing}%)")
print(f"  Both clean                 : {n_both_clean} ({n_both_clean}%)")
print(f"  Label integrity            : PASSED (never changed)")
print(f"  Shape integrity            : PASSED (always preserved)")


## 12. Visualize a fully corrupted sample end-to-end

This is the final check: show exactly what the model will receive during
training — a `MultimodalSample` that may have corrupted image, corrupted text,
or both, along with the corruption metadata flags.


In [ ]:
def visualize_corrupted_sample(sample: MultimodalSample,
                                corruption_module: CorruptionModule,
                                tokenizer, n_examples: int = 4):
    """
    Show n_examples of the same sample with different random corruptions.
    This mimics what the model sees across different training steps.
    """
    fig, axes = plt.subplots(2, n_examples + 1, figsize=((n_examples+1)*3, 7))
    
    # Column 0: clean reference
    axes[0, 0].imshow(denormalize(sample.image))
    axes[0, 0].set_title("Clean\n(reference)", fontsize=9, fontweight="bold")
    axes[0, 0].axis("off")
    axes[1, 0].text(0.05, 0.7, f"Q: {sample.raw_text[:35]}...",
                    fontsize=8, transform=axes[1, 0].transAxes, wrap=True)
    axes[1, 0].axis("off")
    
    # Columns 1+: random corruptions
    for i in range(n_examples):
        corrupted = corruption_module(sample)
        
        # Image row
        axes[0, i+1].imshow(denormalize(corrupted.image))
        img_status = "MISSING" if corrupted.image_missing else                      ("corrupted" if corrupted.image_corrupted else "clean")
        txt_status = "MISSING" if corrupted.text_missing else                      ("corrupted" if corrupted.text_corrupted else "clean")
        axes[0, i+1].set_title(
            f"Example {i+1}\nImg: {img_status}\nTxt: {txt_status}\n"
            f"sev: {corrupted.corruption_severity:.1f}",
            fontsize=8
        )
        axes[0, i+1].axis("off")
        
        # Text row
        decoded = tokenizer.decode(corrupted.text_ids, skip_special_tokens=True)
        decoded = decoded[:60] + "..." if len(decoded) > 60 else decoded
        axes[1, i+1].text(0.05, 0.7, f"Q: {decoded}",
                          fontsize=8, transform=axes[1, i+1].transAxes, wrap=True)
        axes[1, i+1].axis("off")
    
    plt.suptitle("What the model sees during training\n"
                 "(same sample, different random corruptions each step)",
                 fontsize=11)
    plt.tight_layout()
    plt.savefig("corrupted_samples.png", dpi=100, bbox_inches="tight")
    plt.show()
    print("Saved to corrupted_samples.png")

visualize_corrupted_sample(clean_samples[0], corruption_module, tokenizer)


## 13. Save the corruption module as a `.py` file

Now that we've validated everything, we save the corruption module as a proper
Python file. This is the first step in migrating from notebooks to a real
package structure. Notebook 03 onwards will import from this file.


In [ ]:
corruption_code = '''
"""
corruption/corruption_module.py
Corruption module for the ARAF project.
Validated in Notebook 02.
"""

import copy
import random
from typing import Optional, List, Tuple
from dataclasses import dataclass

import torch
import torch.nn.functional as F

# ── Special token IDs (BERT) ──────────────────────────────────────────────────
CLS_ID  = 101
SEP_ID  = 102
PAD_ID  = 0
MASK_ID = 103

SEVERITY_PARAMS = {
    "gaussian_noise" : {1: 0.05, 2: 0.10, 3: 0.20, 4: 0.35, 5: 0.50},
    "motion_blur"    : {1: 3,    2: 5,    3: 7,    4: 11,   5: 15  },
    "occlusion"      : {1: 0.05, 2: 0.10, 3: 0.20, 4: 0.35, 5: 0.50},
    "token_dropout"  : {1: 0.10, 2: 0.20, 3: 0.35, 4: 0.50, 5: 0.70},
    "token_shuffle"  : {1: 0.15, 2: 0.30, 3: 0.50, 4: 0.70, 5: 1.00},
    "token_mask"     : {1: 0.10, 2: 0.20, 3: 0.35, 4: 0.50, 5: 0.70},
}


def get_corruptible_positions(text_ids):
    special = {CLS_ID, SEP_ID, PAD_ID}
    mask = torch.ones(len(text_ids), dtype=torch.bool)
    for i, tid in enumerate(text_ids.tolist()):
        if tid in special:
            mask[i] = False
    return mask.nonzero(as_tuple=True)[0]


def corrupt_gaussian_noise(image, severity):
    std = SEVERITY_PARAMS["gaussian_noise"][severity]
    return (image + torch.randn_like(image) * std).clamp(-3.0, 3.0)


def corrupt_motion_blur(image, severity):
    k = SEVERITY_PARAMS["motion_blur"][severity]
    kernel = (torch.ones(1, 1, 1, k) / k).expand(3, 1, 1, k)
    pad = k // 2
    return F.conv2d(image.unsqueeze(0), kernel, padding=(0, pad),
                    groups=3).squeeze(0)


def corrupt_occlusion(image, severity, num_patches=3):
    fraction = SEVERITY_PARAMS["occlusion"][severity]
    img = image.clone()
    _, H, W = img.shape
    total_area = int(H * W * fraction)
    patch_side = max(int((total_area // num_patches) ** 0.5), 4)
    for _ in range(num_patches):
        top  = random.randint(0, max(0, H - patch_side))
        left = random.randint(0, max(0, W - patch_side))
        img[:, top:top+patch_side, left:left+patch_side] = 0.0
    return img


def corrupt_token_dropout(text_ids, attention_mask, severity):
    fraction = SEVERITY_PARAMS["token_dropout"][severity]
    ids, amask = text_ids.clone(), attention_mask.clone()
    valid_pos = get_corruptible_positions(ids)
    if len(valid_pos) == 0:
        return ids, amask
    n_drop = max(1, int(len(valid_pos) * fraction))
    drop_idx = valid_pos[torch.randperm(len(valid_pos))[:n_drop]]
    ids[drop_idx] = PAD_ID
    amask[drop_idx] = 0
    return ids, amask


def corrupt_token_shuffle(text_ids, attention_mask, severity):
    fraction = SEVERITY_PARAMS["token_shuffle"][severity]
    ids = text_ids.clone()
    valid_pos = get_corruptible_positions(ids)
    if len(valid_pos) == 0:
        return ids, attention_mask
    n_shuffle = max(1, int(len(valid_pos) * fraction))
    shuf_idx = valid_pos[torch.randperm(len(valid_pos))[:n_shuffle]]
    shuffled_order = shuf_idx[torch.randperm(len(shuf_idx))]
    ids[shuf_idx] = ids[shuffled_order].clone()
    return ids, attention_mask


def corrupt_token_mask(text_ids, attention_mask, severity):
    fraction = SEVERITY_PARAMS["token_mask"][severity]
    ids = text_ids.clone()
    valid_pos = get_corruptible_positions(ids)
    if len(valid_pos) == 0:
        return ids, attention_mask
    n_mask = max(1, int(len(valid_pos) * fraction))
    mask_idx = valid_pos[torch.randperm(len(valid_pos))[:n_mask]]
    ids[mask_idx] = MASK_ID
    return ids, attention_mask


def corrupt_missing_modality(sample, drop_image=False, drop_text=False):
    result = copy.deepcopy(sample)
    if drop_image:
        result.image = torch.zeros_like(sample.image)
        result.image_missing = True
        result.image_corrupted = True
    if drop_text:
        result.text_ids = torch.full_like(sample.text_ids, PAD_ID)
        result.attention_mask = torch.zeros_like(sample.attention_mask)
        result.text_ids[0] = CLS_ID
        result.attention_mask[0] = 1
        result.text_missing = True
        result.text_corrupted = True
    result.corruption_severity = 1.0
    return result


class CorruptionModule:
    IMAGE_CORRUPTIONS = {
        "gaussian_noise": corrupt_gaussian_noise,
        "motion_blur":    corrupt_motion_blur,
        "occlusion":      corrupt_occlusion,
    }
    TEXT_CORRUPTIONS = {
        "token_dropout": corrupt_token_dropout,
        "token_shuffle": corrupt_token_shuffle,
        "token_mask":    corrupt_token_mask,
    }

    def __init__(self, p_corrupt_image=0.5, p_corrupt_text=0.5,
                 p_missing_image=0.1, p_missing_text=0.1,
                 severity=None, image_corruptions=None,
                 text_corruptions=None):
        self.p_corrupt_image   = p_corrupt_image
        self.p_corrupt_text    = p_corrupt_text
        self.p_missing_image   = p_missing_image
        self.p_missing_text    = p_missing_text
        self.fixed_severity    = severity
        self.image_corruptions = image_corruptions or list(self.IMAGE_CORRUPTIONS)
        self.text_corruptions  = text_corruptions  or list(self.TEXT_CORRUPTIONS)

    def _sample_severity(self):
        return self.fixed_severity if self.fixed_severity else random.randint(1, 5)

    def __call__(self, sample):
        result = copy.deepcopy(sample)
        sev    = self._sample_severity()
        drop_img  = random.random() < self.p_missing_image
        drop_text = random.random() < self.p_missing_text
        if drop_img or drop_text:
            result = corrupt_missing_modality(result, drop_image=drop_img,
                                              drop_text=drop_text)
            if drop_img and drop_text:
                return result
        if not result.image_missing and random.random() < self.p_corrupt_image:
            fn = self.IMAGE_CORRUPTIONS[random.choice(self.image_corruptions)]
            result.image = fn(result.image, sev)
            result.image_corrupted = True
            result.corruption_severity = sev / 5.0
        if not result.text_missing and random.random() < self.p_corrupt_text:
            fn = self.TEXT_CORRUPTIONS[random.choice(self.text_corruptions)]
            result.text_ids, result.attention_mask = fn(
                result.text_ids, result.attention_mask, sev)
            result.text_corrupted = True
            result.corruption_severity = max(result.corruption_severity, sev / 5.0)
        return result

    def corrupt_batch(self, samples):
        return [self(s) for s in samples]
'''

# Save to file
import os
os.makedirs("corruption", exist_ok=True)
with open("corruption/__init__.py", "w") as f:
    f.write("")
with open("corruption/corruption_module.py", "w") as f:
    f.write(corruption_code)

print("Saved: corruption/corruption_module.py")
print("\nTest import...")
import importlib.util, sys
spec = importlib.util.spec_from_file_location(
    "corruption_module", "corruption/corruption_module.py")
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)
print("Import successful. CorruptionModule ready to use in future notebooks.")


## 14. Summary and what's next

### What we built in this notebook

| Component | What it does |
|---|---|
| `corrupt_gaussian_noise` | Adds Gaussian noise to image tensor |
| `corrupt_motion_blur` | Applies horizontal motion blur via conv |
| `corrupt_occlusion` | Blacks out random rectangular patches |
| `corrupt_token_dropout` | Replaces tokens with [PAD] |
| `corrupt_token_shuffle` | Reorders tokens randomly |
| `corrupt_token_mask` | Replaces tokens with [MASK] |
| `corrupt_missing_modality` | Zeros out entire image or text modality |
| `CorruptionModule` | Unified interface, configurable probabilities |

### Key properties verified

- Labels are never changed by corruption
- Tensor shapes are always preserved
- Special tokens ([CLS], [SEP], [PAD]) are never corrupted
- Missing modality flags are correctly set
- Severity levels produce visually distinct corruption levels

### What Notebook 03 covers

**Baseline models** — before building ARAF, we build simpler models to
compare against:

- **Unimodal image model** — ResNet encoder, ignores text entirely
- **Unimodal text model** — BERT encoder, ignores image entirely
- **Naive fusion model** — concatenates image and text features, no reliability
  weighting

These baselines answer the question: how much does ARAF actually help?
Without baselines, we have no way to measure improvement.
